> **Portfolio summary:** An LLM with a live market-data tool: Gemini calls yfinance to judge whether Apple looks overvalued on P/E.
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

This notebook covers setting up the environment, defining a tool to fetch Apple's P/E ratio using yfinance, configuring a Gemini model with this tool, and then using the model to assess if Apple is overvalued compared to a market P/E of 25. Finally, the chat history was inspected to understand the function calling process.

In [ ]:
# 1. Setup (Use the standard library)
# Get your API key from https://aistudio.google.com/
!pip install -q -U google-generativeai # Ensure the correct library is installed
import google.generativeai as genai # Import the correct module to override previous
from google.colab import userdata
import os

os.environ["GOOGLE_API_KEY"] = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=os.environ["GOOGLE_API_KEY"])

In [ ]:
!pip install -q yfinance

In [ ]:
import yfinance as yf

# Function to get the PE ratio

def get_pe_ratio(ticker: str):

  print(f"... Tool Calling for PE_RATIO for {ticker}....")

  try:
    stock = yf.Ticker(ticker)
    pe = stock.info.get('trailingPE','N/A')
    return pe
  except Exception as e:
    return f"Could not fetch PE ratio of {ticker} due to {e}"


In [ ]:
tool_list = [get_pe_ratio]

# Initialize model with tools

new_model = genai.GenerativeModel(
    "gemini-2.5-flash",
    tools = tool_list
)

# enable automatic function calling

chat = new_model.start_chat(enable_automatic_function_calling=True)

... Tool Calling for PE_RATIO for AAPL
Apple's P/E ratio is 31.36, which is higher than the average market P/E of 25. This suggests that Apple might be overvalued compared to the average market.

In [ ]:

# Find the answer:

response = chat.send_message("Is Apple overvalued compared to the average market P/E of 25 ?")
print(response.text)

Apple's P/E ratio is 31.36. Compared to the average market P/E of 25, Apple appears to be overvalued.

In [ ]:
# 7. Inspection: What actually happened?
# We can look at the chat history to see the "FunctionCall" objects.

print("🕵️‍♂️ HISTORY INSPECTION:\n")
for message in chat.history:
    role = message.role
    print(f"--- {role.upper()} ---")

    # Check if there are function calls
    for part in message.parts:
        if fn := part.function_call:
            print(f"🔧 ACTION: Calling Function '{fn.name}' with args: {dict(fn.args)}")
        elif resp := part.function_response:
            print(f"📨 OBSERVATION: Function returned: {resp.response}")
        else:
            print(part.text)

🕵️‍♂️ HISTORY INSPECTION:

--- USER ---
Is Apple overvalued compared to the average market P/E of 25 ?
--- MODEL ---
🔧 ACTION: Calling Function 'get_pe_ratio' with args: {'ticker': 'AAPL'}
--- USER ---
📨 OBSERVATION: Function returned: <proto.marshal.collections.maps.MapComposite object at 0x795521f15d90>
--- MODEL ---
Apple's P/E ratio is 31.36, which is higher than the average market P/E of 25. This suggests that Apple might be overvalued compared to the average market.
--- USER ---
Is Apple overvalued compared to the average market P/E of 25 ?
--- MODEL ---
Apple's P/E ratio is 31.36. Compared to the average market P/E of 25, Apple appears to be overvalued.